In [1]:
#sudo apt update
#sudo apt install ffmpeg

!pip install whisper-openai
!pip install InaSpeechSegmenter
!pip install pyannote.audio
!pip install --upgrade transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 31.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 87.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 612.9/612.9 kB 16.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 105.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 94.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.6/801.6 kB 13.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 802.1/802.1 kB 82.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23/23 [whisper-openai]m [whisper-openai]]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 71.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 98.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 51.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.8/252.8 MB 77.6 MB/s  0:00:03m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.7/13.7 MB 95.0 MB/s  

### A ne run qu'une seule fois !

In [ ]:
import whisper
import torch

torch.cuda.empty_cache()

whisper_model = whisper.load_model("medium", device="cuda") # remove device for CPU

/opt/python/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████████████████████████████████| 1.42G/1.42G [00:14<00:00, 107MiB/s]


In [ ]:
import os
import s3fs

#A RETIRER SYSTEMATIQUEMENT

#CREDENTIALS


#TOKEN Huggingface
token = ""


fs = s3fs.S3FileSystem(
    client_kwargs={'endpoint_url': 'https://'+'minio.lab.sspcloud.fr'},
    key = os.environ["AWS_ACCESS_KEY_ID"], 
    secret = os.environ["AWS_SECRET_ACCESS_KEY"], 
    token = os.environ["AWS_SESSION_TOKEN"])


# Récupération des fichiers

s3_folder = "s3://lab/PESSD/wav"

wav_files = fs.glob(f"{s3_folder}/*.wav")

for file in wav_files:
    fs.get(file, "data/"+(file.removeprefix("lab/PESSD/wav/")))


# Audios à traiter
file_list = os.listdir("data")

In [ ]:
# Si sélection de fichiers
file_list = ["data/SCR2_wav.wav"]

# Code final

In [11]:
from pyannote.audio import Pipeline
from inaSpeechSegmenter import Segmenter
from inaSpeechSegmenter.export_funcs import seg2csv
import pandas as pd
import os
import whisper

# Pipeline pour diarization
pipeline = Pipeline.from_pretrained(
    'pyannote/speaker-diarization-community-1',
    token=token
)
# remove for CPU friendly
pipeline.to(torch.device("cuda"))

# Modèle InaSpeechSegmenter (genre)
seg_model = Segmenter()

# Liste des fichiers audio
audios = file_list  

#### BOUCLE PRINCIPALE
for audio_path in audios:

    # --- RESET DES LISTES / DATAFRAMES pour ce fichier ---
    diarization_list = []
    merged_list = []
    merged_list_g = []
    df_segments = pd.DataFrame(columns=["numero_segment", "start", "end", "text", "audio_file"])

    # --- TRANSCRIPTION ---
    audio = whisper.load_audio(audio_path)

    # Transcrire avec segmentation
    result = whisper_model.transcribe(audio_path,language="fr",condition_on_previous_text=False,no_speech_threshold=0.6,logprob_threshold=-1.0,compression_ratio_threshold=2.4)

    # Créer une liste de dict pour chaque segment
    segments_data = [
        {
            "start": seg["start"],
            "stop": seg["end"],
            "text": seg["text"].strip(),
            "audio_file": os.path.basename(audio_path)
        }
    for seg in result["segments"]
    ]

    # Transformer en DataFrame
    df_segments = pd.DataFrame(segments_data)

    # --- DIARIZATION ---
    output = pipeline(audio_path)

    for turn, speaker in output.speaker_diarization:
        diarization_list.append({
            "speaker": str(speaker),
            "start": turn.start,
            "end": turn.end
        })

    # Fusionner les segments consécutifs du même locuteur
    for seg_item in diarization_list:
        if not merged_list:
            merged_list.append(seg_item)
        else:
            last = merged_list[-1]
            if seg_item["speaker"] == last["speaker"]:
                last["end"] = seg_item["end"]
            else:
                merged_list.append(seg_item)
    
    ### MAIN SPEAKER
    # Détecter tous les locuteurs
    speakers = sorted({s["speaker"] for s in merged_list})

    # Ajouter colonnes pour chaque speaker
    for spk in speakers:
        df_segments[spk] = 0.0

    # Ajouter colonne pour le locuteur principal
    df_segments["main_speaker"] = None

    # Calcul du temps de parole et du locuteur principal
    for idx, row in df_segments.iterrows():
        seg_start = row["start"]
        seg_end = row["stop"]

    # dictionnaire temporaire pour stocker les durées
        speaker_times = {spk: 0.0 for spk in speakers}

        for s in merged_list:
            # intersection entre le segment et le sous-segment du speaker
            inter_start = max(seg_start, s["start"])
            inter_end = min(seg_end, s["end"])
            duration = max(0.0, inter_end - inter_start)
            if duration > 0:
                speaker_times[s["speaker"]] += duration

    # remplir les colonnes du DataFrame
        for spk, t in speaker_times.items():
            df_segments.at[idx, spk] = t

    # déterminer le locuteur principal
        main_spk = max(speaker_times, key=speaker_times.get)
        df_segments.at[idx, "main_speaker"] = main_spk


    # --- GENRE LOCUTEUR ---
    segmentation = seg_model(audio_path)

    for gender, start, stop in segmentation:
    
        if not merged_list_g:
            merged_list_g.append({"gender": gender, "start": start, "stop": stop})
    
        else:
            last = merged_list_g[-1]
        
            if gender == last["gender"]:
                last["stop"] = stop
        
            else:
                merged_list_g.append({"gender": gender, "start": start, "stop": stop})

    # Détecter tous les locuteurs
    gender = sorted({s["gender"] for s in merged_list_g})

    # Ajouter colonnes pour chaque speaker
    for g in gender:
        df_segments[g] = 0.0

    # Ajouter colonne pour le locuteur principal
    df_segments["main_g"] = None

    # Calcul du temps de parole et du locuteur principal
    for idx, row in df_segments.iterrows():
        seg_start = row["start"]
        seg_end = row["stop"]

        # dictionnaire temporaire pour stocker les durées
        gender_times = {g: 0.0 for g in gender}

        for s in merged_list_g:
            # intersection entre le segment et le sous-segment du speaker
            inter_start = max(seg_start, s["start"])
            inter_end = min(seg_end, s["stop"])
            duration = max(0.0, inter_end - inter_start)
            if duration > 0:
                gender_times[s["gender"]] += duration

        # remplir les colonnes du DataFrame
        for g, t in gender_times.items():
            df_segments.at[idx, g] = t

        # déterminer le locuteur principal
        main_g = max(gender_times, key=gender_times.get)
        df_segments.at[idx, "main_g"] = main_g

    # --- METTRE LE DF AU PROPRE ---
    # Fusionner les lignes d'un même locuteur
    df_segments['bloc'] = (df_segments['main_speaker'] != df_segments['main_speaker'].shift()).cumsum()

    df_segments = df_segments.groupby('bloc').agg(
        start=('start', 'first'),
        stop=('stop', 'last'),
        text=('text', ' '.join),
        main_speaker=('main_speaker', 'first'),
        main_g=('main_g', 'first'),
        audio_file=('audio_file', 'first')

    ).reset_index(drop=True)

    # --- ENREGISTRER LE RESULTAT EN CSV POUR CE FICHIER ---
    csv_name = os.path.splitext(os.path.basename(audio_path))[0] + "_transcription.csv"
    df_segments.to_csv("data/"+csv_name, index=False, encoding="utf-8-sig")
    df_segments.to_csv(f"s3://lab/PESSD/csv/{csv_name}", index=False, encoding="utf-8-sig")
    print(f"CSV créé pour {audio_path} : {csv_name}")

/opt/python/lib/python3.13/site-packages/keras/src/layers/reshaping/reshape.py:38: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/opt/python/lib/python3.13/site-packages/pyannote/audio/utils/reproducibility.py:74: ReproducibilityWarning: TensorFloat-32 (TF32) has been disabled as it might lead to reproducibility issues and lower accuracy.
It can be re-enabled by calling
   >>> import torch
   >>> torch.backends.cuda.matmul.allow_tf32 = True
   >>> torch.backends.cudnn.allow_tf32 = True
See https://github.com/pyannote/pyannote-audio/issues/1370 for more details.

  warnings.warn(
/opt/python/lib/python3.13/site-packages/pyannote/audio/models/blocks/pooling.py:103: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered in

6710/6710 - 19s - 3ms/step
5996/5996 - 36s - 6ms/step
CSV créé pour data/SCR1_wav.wav : SCR1_wav_transcription.csv
